In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
%cd /content
!git clone https://github.com/Chagui05/Proyecto-II-IA.git
%cd Proyecto-II-IA

/content
Cloning into 'Proyecto-II-IA'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 376 (delta 31), reused 75 (delta 20), pack-reused 286 (from 2)
Receiving objects: 100% (376/376), 159.95 MiB | 18.65 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Proyecto-II-IA


In [ ]:
!git pull origin main

From https://github.com/Chagui05/Proyecto-II-IA
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
%cd /content/Proyecto-II-IA
!git fetch origin
!git checkout modelo-b-distillation
!git pull origin modelo-b-distillation

/content/Proyecto-II-IA
Branch 'modelo-b-distillation' set up to track remote branch 'modelo-b-distillation' from 'origin'.
Switched to a new branch 'modelo-b-distillation'
From https://github.com/Chagui05/Proyecto-II-IA
 * branch            modelo-b-distillation -> FETCH_HEAD
Already up to date.


In [ ]:
!pip install lightning hydra-core wandb torchmetrics scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 43.9 MB/s eta 0:00:00


In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: davidblancolascarez (models-instituto-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Entrenamientos Resnet Scratch

In [ ]:
# Esto es para probar que la confif esté bien
!python train.py experiments=resnet_a_baseline --cfg job

In [ ]:
!python train.py experiments=resnet_a_baseline

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_scratch
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_a_baseline
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_a_baseline_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_a_baseline
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${model.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
GPU available: True (cuda), used: True
TPU 

In [ ]:
!python train.py experiments=resnet_a_lr_0005

Streaming output truncated to the last 5000 lines.
Total estimated model params size (MB): 2.737                                   
Modules in train mode: 40                                                       
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:2
1: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, 
TreeSpec) and treespec.is_leaf()` instead.
Epoch 0/99 ━━━━━━━━━━━━━━━━━ 0/71 0:00:00 • -:--:-- 0.00it/s v_num: rrjm        
Epoch 0/99 ━━━━━━━━━━━━━━━━━━━━━━━ 71/71 0:00:01 • 0:00:00 42.74it/s v_num: rrjm
Epoch 0/99 ━━━━━━━━━━━━━━━━━━━━━━━ 71/71 0:00:01 • 0:00:00 42.74it/s v_num: rrjm
Epoch 0/99 ━━━━━━━━━━━━━━━━━━━━━━━ 71/71 0:00:01 • 0:00:00 42.74it/s v_num: rrjm
Epoch 0/99 ━━━━━━━━━━━━━━━━━━━━━━━ 71/71 0:00:01 • 0:00:00 42.74it/s v_num: rrjm
Epoch 0/99 ━━━━━━━

In [ ]:
!python train.py experiments=resnet_a_batch_64

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 64
  num_workers: 2
model:
  name: resnet_scratch
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_a_batch_64
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_a_batch_64_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_a_batch_64
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${model.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
GPU available: True (cuda), used: True
TPU 

In [ ]:
# Con esto se copian los mejores modelos a esa carpeta persistente en drive, ya que el repo es efímero
!cp -r checkpoints/best/ /content/drive/MyDrive/Proyecto2/cache/best_models/

Ya que se ejecutaron los entrenamientos, es momento de evaluar que tan bien detectan anomalias

In [ ]:
!python anomaly_evaluation.py experiments=resnet_a_baseline

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_scratch
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_a_baseline
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_a_baseline_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_a_baseline
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
Matriz de confusión:
[[141 203]
 [393 428]

In [ ]:
!python anomaly_evaluation.py experiments=resnet_a_lr_0005

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_scratch
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.0005
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_a_lr_0005
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_a_lr_0005_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_a_lr_0005
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
Matriz de confusión:
[[121 223]
 [355 466]]


In [ ]:
!python anomaly_evaluation.py experiments=resnet_a_batch_64

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 64
  num_workers: 2
model:
  name: resnet_scratch
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_a_batch_64
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_a_batch_64_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_a_batch_64
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
Matriz de confusión:
[[155 189]
 [406 415]

### Extracción wandb

El siguiente script sirve para extraer todas las métricas de las runs de wandb para poder verlas numéricamente y no solo en gráfico.

In [ ]:
import pandas as pd
import wandb


ENTITY = "sanchaves-instituto-tecnol-gico-de-costa-rica"
PROJECT = "proyecto-2-ia"

RUN_NAMES = [
    "resnet_a_baseline",
    "resnet_a_lr_0005",
    "resnet_a_batch_64",
]


def get_run_by_name(api, entity, project, run_name):
    runs = api.runs(f"{entity}/{project}")
    for run in runs:
        if run.name == run_name:
            return run

    raise ValueError(f"No se encontró el run: {run_name}")


def last_non_null(df, column):
    if column not in df.columns:
        return None

    values = df[column].dropna()
    if values.empty:
        return None

    return values.iloc[-1]


def value_at_or_before_step(df, column, step):
    if column not in df.columns or "_step" not in df.columns:
        return None

    filtered = df[df["_step"] <= step]
    values = filtered[column].dropna()

    if values.empty:
        return None

    return values.iloc[-1]


def summarize_run(run_name, history):
    summary = {
        "run_name": run_name,
    }

    if "val/loss" in history.columns:
        val_loss_rows = history[history["val/loss"].notna()]
        best_row = val_loss_rows.loc[val_loss_rows["val/loss"].idxmin()]

        best_step = best_row["_step"]

        summary["best_val_loss"] = best_row["val/loss"]
        summary["best_val_loss_step"] = best_step
        summary["best_val_loss_epoch"] = best_row["epoch"] if "epoch" in history.columns else None

        summary["val_acc_at_best_val_loss"] = best_row["val/acc"] if "val/acc" in history.columns else None
        summary["train_loss_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/loss",
            best_step,
        )
        summary["train_acc_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/acc",
            best_step,
        )
    else:
        summary["best_val_loss"] = None
        summary["best_val_loss_step"] = None
        summary["best_val_loss_epoch"] = None
        summary["val_acc_at_best_val_loss"] = None
        summary["train_loss_at_best_val_loss"] = None
        summary["train_acc_at_best_val_loss"] = None

    summary["final_train_loss"] = last_non_null(history, "train/loss")
    summary["final_val_loss"] = last_non_null(history, "val/loss")
    summary["final_train_acc"] = last_non_null(history, "train/acc")
    summary["final_val_acc"] = last_non_null(history, "val/acc")

    if "val/acc" in history.columns:
        summary["best_val_acc"] = history["val/acc"].max()
    else:
        summary["best_val_acc"] = None

    if summary["final_train_acc"] is not None and summary["final_val_acc"] is not None:
        summary["final_acc_gap"] = summary["final_train_acc"] - summary["final_val_acc"]
    else:
        summary["final_acc_gap"] = None

    if summary["final_train_loss"] is not None and summary["final_val_loss"] is not None:
        summary["final_loss_gap"] = summary["final_val_loss"] - summary["final_train_loss"]
    else:
        summary["final_loss_gap"] = None

    return summary

In [ ]:
api = wandb.Api()

summaries = []

for run_name in RUN_NAMES:
    print(f"\nExtrayendo métricas de: {run_name}")

    run = get_run_by_name(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        run_name=run_name,
    )

    rows = list(run.scan_history())
    history = pd.DataFrame(rows)

    summary = summarize_run(run_name, history)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)

print("\nResumen de entrenamientos:")
print(summary_df.to_markdown(index=False))


Extrayendo métricas de: resnet_a_baseline

Extrayendo métricas de: resnet_a_lr_0005

Extrayendo métricas de: resnet_a_batch_64

Resumen de entrenamientos:
| run_name          |   best_val_loss |   best_val_loss_step |   best_val_loss_epoch |   val_acc_at_best_val_loss |   train_loss_at_best_val_loss |   train_acc_at_best_val_loss |   final_train_loss |   final_val_loss |   final_train_acc |   final_val_acc |   best_val_acc |   final_acc_gap |   final_loss_gap |
|:------------------|----------------:|---------------------:|----------------------:|---------------------------:|------------------------------:|-----------------------------:|-------------------:|-----------------:|------------------:|----------------:|---------------:|----------------:|-----------------:|
| resnet_a_baseline |     0.000465465 |                   10 |                     4 |                          1 |                     0.0355495 |                     0.991971 |         0.0209189  |      0.00148938  |    

## Entrenamientos Resnet teacher-student

#### Etapa 1: entrenar el teacher


In [ ]:
!python train.py experiments=resnet_teacher --cfg job   # opcional: revisar config

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_teacher
  num_classes: 10
  image_size: 128
  pretrained: true
  lr: 0.0001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_teacher
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_teacher_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_teacher
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}
  device: cuda
  percentile: 95
seed: 42


In [ ]:
!python train.py experiments=resnet_teacher

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_teacher
  num_classes: 10
  image_size: 128
  pretrained: true
  lr: 0.0001
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_teacher
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_teacher_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_teacher
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}
  device: cuda
  percentile: 95
seed: 42

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 

#### Etapa 2: entrenamiento de 3 students

In [ ]:
!python train.py experiments=resnet_b_baseline

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_distill
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
  temperature: 4.0
  alpha: 0.7
  teacher_checkpoint_path: checkpoints/best/resnet_teacher_best.ckpt
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_b_baseline
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_b_baseline_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_b_baseline
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name

In [ ]:
!python train.py experiments=resnet_b_T2_alpha05

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_distill
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
  temperature: 2.0
  alpha: 0.5
  teacher_checkpoint_path: checkpoints/best/resnet_teacher_best.ckpt
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_b_T2_alpha05
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_b_T2_alpha05_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_b_T2_alpha05
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logge

In [ ]:
!python train.py experiments=resnet_b_lr_0005

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_distill
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.0005
  temperature: 4.0
  alpha: 0.7
  teacher_checkpoint_path: checkpoints/best/resnet_teacher_best.ckpt
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_b_lr_0005
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_b_lr_0005_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_b_lr_0005
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logger.name}


#### Copiar los mejores modelos a Drive

In [ ]:
!cp -r checkpoints/best/ /content/drive/MyDrive/Proyecto2/cache/best_models/

#### Evaluar anomalías

In [ ]:
!python anomaly_evaluation.py experiments=resnet_b_baseline

Traceback (most recent call last):
  File "/content/Proyecto-II-IA/anomaly_evaluation.py", line 3, in <module>
    import hydra
ModuleNotFoundError: No module named 'hydra'


In [ ]:
!python anomaly_evaluation.py experiments=resnet_b_T2_alpha05

data:
  checkpoint_path: /content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt
  batch_size: 32
  num_workers: 2
model:
  name: resnet_distill
  in_channels: 3
  image_size: 128
  num_classes: 10
  base_channels: 64
  lr: 0.001
  temperature: 2.0
  alpha: 0.5
  teacher_checkpoint_path: checkpoints/best/resnet_teacher_best.ckpt
trainer:
  max_epochs: 100
  accelerator: gpu
  devices: auto
  check_val_every_n_epoch: 1
  log_every_n_steps: 10
  deterministic: true
  benchmark: false
  early_stopping:
    monitor: val/loss
    mode: min
    patience: 10
    min_delta: 0.0001
  checkpoint:
    monitor: val/loss
    mode: min
    save_top_k: 1
    dirpath: checkpoints/resnet_b_T2_alpha05
    filename: ${model.name}-{epoch:02d}
    export_path: checkpoints/best/resnet_b_T2_alpha05_best.ckpt
logger:
  project: proyecto-2-ia
  name: resnet_b_T2_alpha05
  log_model: false
evaluation:
  checkpoint_path: ${trainer.checkpoint.export_path}
  output_dir: outputs/anomaly_evaluation/${logge

In [ ]:
!python anomaly_evaluation.py experiments=resnet_b_lr_0005

Traceback (most recent call last):
  File "/content/Proyecto-II-IA/anomaly_evaluation.py", line 3, in <module>
    import hydra
ModuleNotFoundError: No module named 'hydra'


#### Visualizar metricas

In [ ]:
import pandas as pd
import wandb


ENTITY = "models-instituto-tecnol-gico-de-costa-rica"
PROJECT = "proyecto-2-ia"

RUN_NAMES = [
    "resnet_teacher",
    "resnet_b_baseline",
    "resnet_b_T2_alpha05",
    "resnet_b_lr_0005",
]


def get_run_by_name(api, entity, project, run_name):
    runs = api.runs(f"{entity}/{project}")
    for run in runs:
        if run.name == run_name:
            return run

    raise ValueError(f"No se encontró el run: {run_name}")


def last_non_null(df, column):
    if column not in df.columns:
        return None

    values = df[column].dropna()
    if values.empty:
        return None

    return values.iloc[-1]


def value_at_or_before_step(df, column, step):
    if column not in df.columns or "_step" not in df.columns:
        return None

    filtered = df[df["_step"] <= step]
    values = filtered[column].dropna()

    if values.empty:
        return None

    return values.iloc[-1]


def summarize_run(run_name, history):
    summary = {
        "run_name": run_name,
    }

    if "val/loss" in history.columns:
        val_loss_rows = history[history["val/loss"].notna()]
        best_row = val_loss_rows.loc[val_loss_rows["val/loss"].idxmin()]

        best_step = best_row["_step"]

        summary["best_val_loss"] = best_row["val/loss"]
        summary["best_val_loss_step"] = best_step
        summary["best_val_loss_epoch"] = best_row["epoch"] if "epoch" in history.columns else None

        summary["val_acc_at_best_val_loss"] = best_row["val/acc"] if "val/acc" in history.columns else None
        summary["train_loss_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/loss",
            best_step,
        )
        summary["train_acc_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/acc",
            best_step,
        )
        # Métricas propias del destilado (solo existen para el Modelo B / student).
        summary["train_ce_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/ce",
            best_step,
        )
        summary["train_kd_at_best_val_loss"] = value_at_or_before_step(
            history,
            "train/kd",
            best_step,
        )
    else:
        summary["best_val_loss"] = None
        summary["best_val_loss_step"] = None
        summary["best_val_loss_epoch"] = None
        summary["val_acc_at_best_val_loss"] = None
        summary["train_loss_at_best_val_loss"] = None
        summary["train_acc_at_best_val_loss"] = None
        summary["train_ce_at_best_val_loss"] = None
        summary["train_kd_at_best_val_loss"] = None

    summary["final_train_loss"] = last_non_null(history, "train/loss")
    summary["final_val_loss"] = last_non_null(history, "val/loss")
    summary["final_train_acc"] = last_non_null(history, "train/acc")
    summary["final_val_acc"] = last_non_null(history, "val/acc")

    # Componentes del destilado al final del entrenamiento (None para el teacher).
    summary["final_train_ce"] = last_non_null(history, "train/ce")
    summary["final_train_kd"] = last_non_null(history, "train/kd")

    if "val/acc" in history.columns:
        summary["best_val_acc"] = history["val/acc"].max()
    else:
        summary["best_val_acc"] = None

    if summary["final_train_acc"] is not None and summary["final_val_acc"] is not None:
        summary["final_acc_gap"] = summary["final_train_acc"] - summary["final_val_acc"]
    else:
        summary["final_acc_gap"] = None

    if summary["final_train_loss"] is not None and summary["final_val_loss"] is not None:
        summary["final_loss_gap"] = summary["final_val_loss"] - summary["final_train_loss"]
    else:
        summary["final_loss_gap"] = None

    return summary


api = wandb.Api()

summaries = []

for run_name in RUN_NAMES:
    print(f"\nExtrayendo métricas de: {run_name}")

    run = get_run_by_name(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        run_name=run_name,
    )

    rows = list(run.scan_history())
    history = pd.DataFrame(rows)

    summary = summarize_run(run_name, history)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)

print("\nResumen de entrenamientos:")
print(summary_df.to_markdown(index=False))


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



Extrayendo métricas de: resnet_teacher

Extrayendo métricas de: resnet_b_baseline

Extrayendo métricas de: resnet_b_T2_alpha05

Extrayendo métricas de: resnet_b_lr_0005

Resumen de entrenamientos:
| run_name            |   best_val_loss |   best_val_loss_step |   best_val_loss_epoch |   val_acc_at_best_val_loss |   train_loss_at_best_val_loss |   train_acc_at_best_val_loss |   train_ce_at_best_val_loss |   train_kd_at_best_val_loss |   final_train_loss |   final_val_loss |   final_train_acc |   final_val_acc |   final_train_ce |   final_train_kd |   best_val_acc |   final_acc_gap |   final_loss_gap |
|:--------------------|----------------:|---------------------:|----------------------:|---------------------------:|------------------------------:|-----------------------------:|----------------------------:|----------------------------:|-------------------:|-----------------:|------------------:|----------------:|-----------------:|-----------------:|---------------:|----------------:|

#### Evaluación con AUROC:

Las métricas anteriores (F1, accuracy) dependen de un umbral fijo (percentil 95), por lo que mezclan la calidad del embedding con la calibración de la frontera. Esto engaña: un modelo que marca todo como anomalía logra F1 ≈ 0.83 solo porque el ~70% del test son anomalías, sin habilidad real.

Usamos AUROC para aislar la capacidad discriminativa del embedding:

* Es independiente del umbral (evalúa el ranking completo de scores).
* 0.5 = detector aleatorio, 1.0 = separación perfecta.

Así el F1 mide el desempeño operativo y el AUROC revela si el embedding contiene información de anomalía, sin importar dónde se fije el umbral.

In [ ]:
import torch
from sklearn.metrics import roc_auc_score

from data.datamodule import MVTecDataModule
from models.resnet_scratch.model import ResNetScratchClassifier
from models.resnet_distill.model import ResNetDistillClassifier
from utils.evaluation_helpers import evaluate_model_with_mahalanobis

DATA = "/content/drive/MyDrive/Proyecto2/cache/mvtec_anomaly_detection.pt"
CKPT = "/content/drive/MyDrive/Proyecto2/cache/best_models/best"

dm = MVTecDataModule(checkpoint_path=DATA, batch_size=32, num_workers=2)
dm.prepare_data()
dm.setup(stage=None)

# (tipo, ruta del checkpoint) — ajusta los nombres si el ls muestra otros
MODELS = {
    "resnet_teacher": ("teacher", f"{CKPT}/resnet_teacher_best.ckpt"),
    "resnet_a_baseline":   ("scratch", f"{CKPT}/resnet_a_baseline_best.ckpt"),
    "resnet_b_baseline":   ("distill", f"{CKPT}/resnet_b_baseline_best.ckpt"),
    "resnet_b_T2_alpha05": ("distill", f"{CKPT}/resnet_b_T2_alpha05_best.ckpt"),
    "resnet_b_lr_0005":    ("distill", f"{CKPT}/resnet_b_lr_0005_best.ckpt"),
}

print(f"{'modelo':<22} {'AUROC':>7} {'F1':>7}")
for name, (kind, path) in MODELS.items():
    if kind == "scratch":
        model = ResNetScratchClassifier.load_from_checkpoint(path)
    elif kind == "teacher":
        model = ResNet18Teacher.load_from_checkpoint(path)
    else:
        model = ResNetDistillClassifier.load_from_checkpoint(path)

    ev = evaluate_model_with_mahalanobis(
        model=model,
        val_dataloader=dm.val_dataloader(),
        test_dataloader=dm.test_dataloader(),
        device="cuda",
        percentile=95,
        model_name=name,
    )
    y_true = (ev["y_test"][:, 1] != 0).long().numpy()   # 1 = anomalía
    auroc = roc_auc_score(y_true, ev["scores"])
    print(f"{name:<22} {auroc:>7.3f} {ev['results']['f1']:>7.3f}")


modelo                   AUROC      F1
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 66.9MB/s]


Matriz de confusión:
[[  0 344]
 [  0 821]]

Reporte:
              precision    recall  f1-score   support

        good       0.00      0.00      0.00       344
     anomaly       0.70      1.00      0.83       821

    accuracy                           0.70      1165
   macro avg       0.35      0.50      0.41      1165
weighted avg       0.50      0.70      0.58      1165

resnet_teacher           0.678   0.827
Matriz de confusión:
[[141 203]
 [393 428]]

Reporte:
              precision    recall  f1-score   support

        good       0.26      0.41      0.32       344
     anomaly       0.68      0.52      0.59       821

    accuracy                           0.49      1165
   macro avg       0.47      0.47      0.46      1165
weighted avg       0.56      0.49      0.51      1165

resnet_a_baseline        0.480   0.590
Matriz de confusión:
[[ 97 247]
 [288 533]]

Reporte:
              precision    recall  f1-score   support

        good       0.25      0.28      0.27       3

#### Análisis de resultados:

| Modelo                 | AUROC     | Lectura                                                                               |
| ---------------------- | --------- | ------------------------------------------------------------------------------------- |
| resnet_teacher (512-d) | 0.68      | Señal real; su F1 alto es engañoso (marca todo como anomalía → umbral mal calibrado). |
| resnet_b (128-d)       | 0.51–0.52 | Apenas sobre el azar.                                                                 |
| resnet_a (128-d)       | 0.48      | Equivalente a aleatorio.                                                              |
Conclusiones:

Los embeddings del backbone pequeño (A y B) son detectores casi aleatorios (AUROC ≈ 0.5). Al entrenarse solo con datos good y aplicar global average pooling, el embedding codifica la clase del objeto, no la presencia de defectos (a menudo pequeños y locales).

El destilado aporta una mejora pequeña pero consistente: A (0.48) → B (0.51–0.52) en las tres variantes. El student absorbió algo de la estructura del teacher.

El teacher (ResNet-18 completo) sí tiene señal (0.68): la información de anomalía existe en su embedding, pero se pierde al comprimirla en los 128-d del student. La destilación transfiere la capacidad de clasificar, pero no todo el conocmiento necesario para detectar anomalías.

## Entrenamientos U-net